In [ ]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import sys

sys.path.insert(0, "/home/joshua/PhD_year_1/jaxsp/Adding_stellar_masses")

import jaxsp as jsp

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import jax.numpy as jnp

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from jaxsp.constants import GN, hbar

import Stellar_sim_funcs as SSF

from collections import defaultdict

from scipy.special import sph_harm_y

In [ ]:
def construct_rho_rtp(total_mass, R_j_r_phased, aj, parent_j, Y_lm):

    R_modes = R_j_r_phased[:, parent_j]  # (Nr, Nmodes)
    aj_modes = aj[parent_j]  # (Nmodes,)

    full_psi_rtp = jnp.einsum('k,rk,ktp->rtp', aj_modes, R_modes, Y_lm)

    psi_abs2 = jnp.abs(full_psi_rtp) ** 2
    rho_rtp = total_mass * psi_abs2

    return rho_rtp


def precompute_lm_pairs_Ylms(l):

    '''Precompute (l, m) pair for spherical harmonics'''

    lm_l = []      # list of l for each mode k
    lm_m = []      # list of m for each mode k
    parent_j = []  # which radial eigenstate j this (l,m) mode comes from
    lm_pairs = defaultdict(int)

    for j_idx, ell in enumerate(l.tolist()):
        for m in range(-ell, ell + 1):
            lm_l.append(ell)
            lm_m.append(m)
            parent_j.append(j_idx)
            lm_pairs[(ell, m)] += 1

    lm_pairs = list(lm_pairs.keys()) # list of ((l,m), count) pairs

    lm_pairs = jnp.array(lm_pairs)  # shape (Nmodes, 2)


    '''Precompute Y_lm's for wavefunction reconstruction'''

    # McEwen-Wiaux-style equiangular grid

    L = max(l)+1

    L_max_out = 2 * L - 1

    n_theta = L_max_out
    n_phi = 2 * L_max_out - 1

    # Generate theta values
    i = jnp.arange(n_theta)
    theta = (jnp.pi * (2 * i + 1)) / (2 * L_max_out - 1)
    # Generate phi values
    j = jnp.arange(n_phi)
    phi = (2 * jnp.pi * j) / (2 * L_max_out - 1)


    Theta, Phi = jnp.meshgrid(theta, phi, indexing="ij")  # both (n_theta, n_phi)

    Y_list = []
    for ell, m in zip(lm_l, lm_m):
        Y_lm_mode = sph_harm_y(ell, m, Theta, Phi)  # (n_theta, n_phi), complex
        Y_list.append(Y_lm_mode)

    Y_lm = jnp.stack(Y_list, axis=0)  # (Nmodes, n_theta, n_phi), complex


    return jnp.array(parent_j), Y_lm, lm_pairs, jnp.array(lm_l), jnp.array(lm_m), theta, phi

In [ ]:

m22 = 1
u = jsp.set_schroedinger_units(m22)
r_max_enclosing_frac = 0.99
Nt = 1000
total_time = 2.5
dt = total_time * u.from_Gyr / Nt


cNFWtides_params = jnp.array([
357964808.148399 * u.from_Msun,
25.690207,
0.407461,
0.012670 * u.from_Kpc,
1.857991 * u.from_Kpc,
3.729259
])

density_params = jsp.init_core_NFW_tides_params_from_sample(cNFWtides_params)

N = 512
rmin = .1 * u.from_pc
rmax = jsp.enclosing_radius(0.999, density_params)
potential_params = jsp.init_potential_params(density_params, rmin, rmax, N)

eval_library = jax.vmap(jax.vmap(jsp.eval_radial_eigenmode, in_axes=(None, 0)), in_axes=(0,None))

N = 1024
a = 1
b = 10

rmax = jsp.enclosing_radius(r_max_enclosing_frac, density_params)
eigenstate_lib = jsp.init_eigenstate_library(potential_params, rmin, rmax, a, b, N)

l = eigenstate_lib.radial_eigenmode_params.l
eigen_energies = eigenstate_lib.radial_eigenmode_params.E


tol = 1e-7
wavefunction_params = jsp.init_wavefunction_params(eigenstate_lib, density_params, rmin, rmax, tol)


aj_2 = wavefunction_params.aj_2        # shape (Nj,)
rand_phase = jax.random.uniform(jax.random.PRNGKey(0), shape=aj_2.shape, minval=0.0, maxval=2 * jnp.pi,)
aj = jnp.sqrt(aj_2) * jnp.exp(1j * rand_phase)  # shape (Nj,)

total_mass = wavefunction_params.total_mass

print('l max from jaxsp:', max(l))
L = int(max(l) + 1)


parent_j, Y_lm, lm_pairs, lm_l_per_mode, lm_m_per_mode, theta, phi = precompute_lm_pairs_Ylms(l)



r_bins = np.linspace(10, 500, 2)
r_arrays = [np.logspace(np.log10(rmin), np.log10(rmax), int(r_bins[i])) for i in range(len(r_bins))]

for r in r_arrays:

    R_j_r = eval_library(r, eigenstate_lib.radial_eigenmode_params)  # (Nr, Nj)
    R_j_r_fixed = R_j_r

    phase = jnp.exp(-1j * eigen_energies * 1 * dt)
    R_j_r_phased = R_j_r_fixed * phase[None, :]

    rho_rtp = construct_rho_rtp(total_mass, R_j_r_phased, aj, parent_j, Y_lm)  # (Nr, n_theta, n_phi)

    M_enc = SSF.Enclosed_mass_3d(r, theta, phi, rho_rtp, r)
    plt.plot(r * float(u.to_Kpc), M_enc * float(u.to_Msun), label=f'{len(r)} radial bins', alpha = 0.4)

plt.xscale('log')
plt.yscale('log')
plt.legend()
plt.show()